# 02 — Data Preparation Pipeline
## Hackathon IBM — Détection de Fraude Bancaire

**But unique de ce notebook** : transformer chaque fichier `.parquet` brut (1 test + N train undersamplés) en un fichier `prepared_*.parquet` prêt pour l'entraînement de **CatBoost**, **XGBoost** et **TabNet**.

**Ce notebook NE fait PAS** : aucun entraînement de modèle, aucune split train/val, aucune évaluation. Uniquement du feature engineering reproductible et idempotent.

**Contrainte hackathon** : le modèle sera évalué sur des **clients inconnus** → toutes les features sont conçues pour être calculables sur un client qui n'existe pas dans le train (profil statique + velocity à l'intérieur du fichier courant).

### Pipeline appliqué à chaque fichier
1. Nettoyage de base ($ → float, `Yes/No` → 1/0, `date` → datetime)
2. Features temporelles (hour, dayofweek, month, is_weekend, is_night)
3. Features géographiques (home_state via nearest US centroid, is_out_of_state)
4. Features de vélocité (time_since_last_trans_min, amount_last_24h, n_tx_last_24h)
5. Ratios financiers (ratio_amount_credit_limit, ratio_amount_monthly_income)
6. Golden features (previous_transaction_had_error, is_micro_transaction, is_round_amount)
7. Typage final (float32 pour num, category pour cat)
8. Drop des identifiants et colonnes brutes

---
## 0. Imports & configuration

In [ ]:
import os
import re
import glob
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

INPUT_DIR  = Path('.')
OUTPUT_DIR = Path('.')
OUTPUT_PREFIX = 'prepared_'

---
## 1. Constantes & référentiels

On code en dur les centroïdes des 51 états US (lat/long) pour l'assignation `home_state` — méthode déterministe, généralisable à n'importe quel client et ne nécessitant aucun téléchargement externe.

In [ ]:
US_STATE_CENTROIDS = {
    'AL': (32.806671, -86.791130), 'AK': (61.370716, -152.404419), 'AZ': (33.729759, -111.431221),
    'AR': (34.969704, -92.373123),  'CA': (36.116203, -119.681564), 'CO': (39.059811, -105.311104),
    'CT': (41.597782, -72.755371),  'DE': (39.318523, -75.507141),  'DC': (38.897438, -77.026817),
    'FL': (27.766279, -81.686783),  'GA': (33.040619, -83.643074),  'HI': (21.094318, -157.498337),
    'ID': (44.240459, -114.478828), 'IL': (40.349457, -88.986137),  'IN': (39.849426, -86.258278),
    'IA': (42.011539, -93.210526),  'KS': (38.526600, -96.726486),  'KY': (37.668140, -84.670067),
    'LA': (31.169546, -91.867805),  'ME': (44.693947, -69.381927),  'MD': (39.063946, -76.802101),
    'MA': (42.230171, -71.530106),  'MI': (43.326618, -84.536095),  'MN': (45.694454, -93.900192),
    'MS': (32.741646, -89.678696),  'MO': (38.456085, -92.288368),  'MT': (46.921925, -110.454353),
    'NE': (41.125370, -98.268082),  'NV': (38.313515, -117.055374), 'NH': (43.452492, -71.563896),
    'NJ': (40.298904, -74.521011),  'NM': (34.840515, -106.248482), 'NY': (42.165726, -74.948051),
    'NC': (35.630066, -79.806419),  'ND': (47.528912, -99.784012),  'OH': (40.388783, -82.764915),
    'OK': (35.565342, -96.928917),  'OR': (44.572021, -122.070938), 'PA': (40.590752, -77.209755),
    'RI': (41.680893, -71.511780),  'SC': (33.856892, -80.945007),  'SD': (44.299782, -99.438828),
    'TN': (35.747845, -86.692345),  'TX': (31.054487, -97.563461),  'UT': (40.150032, -111.862434),
    'VT': (44.045876, -72.710686),  'VA': (37.769337, -78.169968),  'WA': (47.400902, -121.490494),
    'WV': (38.491226, -80.954453),  'WI': (44.268543, -89.616508),  'WY': (42.755966, -107.302490),
}

# Colonnes monétaires à convertir depuis des strings '$1,234.56'
MONEY_COLS = ['amount', 'credit_limit', 'yearly_income', 'total_debt', 'per_capita_income']

# Colonnes booléennes texte à convertir en int
BOOL_COLS = ['has_chip', 'card_on_dark_web']

# Identifiants à SUPPRIMER (anti-leakage, force la généralisation)
ID_COLS = ['client_id', 'card_id', 'transaction_id', 'merchant_id', 'card_number', 'cvv']

# Colonnes brutes à supprimer après extraction des features dérivées
RAW_COLS_TO_DROP = ['date', 'datetime', 'expires', 'acct_open_date',
                    'latitude', 'longitude', 'zip', 'address',
                    'birth_year', 'birth_month', 'year',
                    'dow', 'day_name',           # doublons créés en amont par l'EDA
                    'amount', 'credit_limit', 'yearly_income',
                    'total_debt', 'per_capita_income',
                    'amount_clean', 'credit_limit_clean', 'yearly_income_clean',
                    'total_debt_clean', 'per_capita_income_clean',
                    'mcc',                       # on garde mcc_description (plus riche)
                    'mcc_desc',                  # sera renommé en mcc_description
                    'merchant_state_clean', 'home_state']  # internes, remplacés par is_out_of_state

# Colonnes CATÉGORIELLES finales (dtype='category' pour CatBoost/LightGBM)
CATEGORICAL_COLS = ['mcc_description', 'merchant_state', 'merchant_city',
                    'use_chip', 'card_type', 'card_brand',
                    'gender', 'errors', 'geo_bucket']

---
## 2. Helpers bas niveau

In [ ]:
def clean_money_column(series: pd.Series) -> pd.Series:
    """'$1,234.56' -> 1234.56 (float32). Robuste aux NaN et aux colonnes déjà numériques."""
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(np.float32)
    return (
        series.astype(str)
        .str.replace(r'[\$,]', '', regex=True)
        .str.strip()
        .replace({'': np.nan, 'nan': np.nan, 'None': np.nan, 'NaN': np.nan})
        .astype(np.float32)
    )


def yes_no_to_int(series: pd.Series) -> pd.Series:
    """'Yes'/'YES'/'yes' -> 1, autres -> 0. NaN -> 0."""
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(np.int8)
    return (series.astype(str).str.strip().str.lower() == 'yes').astype(np.int8)


def latlong_to_state(lat_arr: np.ndarray, lon_arr: np.ndarray) -> np.ndarray:
    """Affecte à chaque (lat, lon) le code d'état US du centroïde le plus proche (haversine vectorisée)."""
    states = np.array(list(US_STATE_CENTROIDS.keys()))
    centroids = np.deg2rad(np.array(list(US_STATE_CENTROIDS.values())))
    pts = np.deg2rad(np.c_[lat_arr.astype(float), lon_arr.astype(float)])
    dlat = pts[:, [0]] - centroids[:, 0]
    dlon = pts[:, [1]] - centroids[:, 1]
    a = np.sin(dlat / 2) ** 2 + np.cos(pts[:, [0]]) * np.cos(centroids[:, 0]) * np.sin(dlon / 2) ** 2
    d = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    out = states[np.argmin(d, axis=1)]
    # Si lat/lon étaient NaN, renvoie 'UNK'
    nan_mask = np.isnan(lat_arr) | np.isnan(lon_arr)
    out = np.where(nan_mask, 'UNK', out)
    return out


def memory_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1024**2


def safe_divide(num: pd.Series, den: pd.Series) -> pd.Series:
    """num / den en remplaçant les 0 du dénominateur par NaN, et les inf par NaN.
    Les arbres (XGBoost, CatBoost, LightGBM) gèrent nativement les NaN."""
    den_safe = den.replace(0, np.nan)
    out = num / den_safe
    return out.replace([np.inf, -np.inf], np.nan).astype(np.float32)

---
## 3. Sous-fonctions par bloc de features

Chaque sous-fonction est **pure** (prend un df, renvoie un df modifié) et **idempotente** (ne casse rien si elle est rejouée). Cette modularité permet de tester chaque bloc individuellement et de réutiliser le code en production.

In [ ]:
def _basic_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    """Étape 1 — Nettoyage de base : $, Yes/No, datetime.
    Robuste : si une version _clean existe déjà, on l'utilise en priorité."""

    for col in MONEY_COLS:
        clean_col = f'{col}_clean'
        if clean_col in df.columns and pd.api.types.is_numeric_dtype(df[clean_col]):
            df[col] = df[clean_col].astype(np.float32)
        elif col in df.columns:
            df[col] = clean_money_column(df[col])

    for col in BOOL_COLS:
        if col in df.columns:
            df[col] = yes_no_to_int(df[col])

    # Datetime : priorité à 'datetime' (déjà typé en amont), sinon parse 'date'.
    if 'datetime' in df.columns and pd.api.types.is_datetime64_any_dtype(df['datetime']):
        df['_ts'] = df['datetime']
    elif 'date' in df.columns:
        df['_ts'] = pd.to_datetime(df['date'], errors='coerce')
    else:
        df['_ts'] = pd.NaT

    # Nettoyage texte des colonnes catégorielles
    if 'errors' in df.columns:
        df['errors'] = df['errors'].fillna('none').astype(str).str.lower().str.strip()
        df['has_error'] = (df['errors'] != 'none').astype(np.int8)

    if 'merchant_state' in df.columns:
        df['merchant_state'] = (df['merchant_state'].astype(str).str.upper().str.strip()
                                .replace({'NAN': np.nan, 'NONE': np.nan, '': np.nan}))

    # amount_abs et is_refund dérivés utiles partout en aval
    if 'amount' in df.columns:
        df['amount_abs'] = df['amount'].abs().astype(np.float32)
        df['is_refund']  = (df['amount'] < 0).astype(np.int8)

    # mcc_description : priorité à mcc_description, sinon mcc_desc, sinon fallback sur mcc numérique
    if 'mcc_description' not in df.columns:
        if 'mcc_desc' in df.columns:
            df['mcc_description'] = df['mcc_desc'].astype(str)
        elif 'mcc' in df.columns:
            df['mcc_description'] = df['mcc'].astype(str)

    return df

In [ ]:
def _time_features(df: pd.DataFrame) -> pd.DataFrame:
    """Étape 2 — Features temporelles."""
    ts = df['_ts']
    df['hour']       = ts.dt.hour.astype('Int16')
    df['dayofweek']  = ts.dt.dayofweek.astype('Int16')   # 0 = lundi
    df['month']      = ts.dt.month.astype('Int16')
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(np.int8)
    df['is_night']   = df['hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(np.int8)
    return df

In [ ]:
def _geo_features(df: pd.DataFrame) -> pd.DataFrame:
    """Étape 3 — Features géographiques. Nécessite latitude/longitude du client."""

    if {'latitude', 'longitude'}.issubset(df.columns):
        lat = df['latitude'].astype(float).values
        lon = df['longitude'].astype(float).values
        df['home_state'] = latlong_to_state(lat, lon)
    else:
        df['home_state'] = 'UNK'

    df['is_online_tx'] = (
        df.get('use_chip', pd.Series('', index=df.index))
          .astype(str).str.lower().str.contains('online')
    ).astype(np.int8)

    ms = df.get('merchant_state', pd.Series(np.nan, index=df.index))

    df['is_out_of_state'] = np.where(
        df['is_online_tx'] == 1, 1,
        np.where(ms.isna(), 1,
                 (ms != df['home_state']).astype(np.int8))
    ).astype(np.int8)

    # Bucket géographique à 4 niveaux (plus riche que le binaire)
    df['geo_bucket'] = np.select(
        [df['is_online_tx'] == 1,
         ms.isna(),
         ms == df['home_state']],
        ['online', 'unknown_state', 'in_state'],
        default='out_of_state'
    )

    return df

In [ ]:
def _velocity_features(df: pd.DataFrame) -> pd.DataFrame:
    """Étape 4 — Velocity sur le grain CARD (pas client — un fraudeur cible une carte précise).

    Features :
      - time_since_last_trans_min : minutes depuis la dernière tx de la même carte
      - amount_last_24h           : somme des montants sur les 24h précédentes (tx exclue)
      - n_tx_last_24h             : nb de tx sur les 24h précédentes
      - previous_transaction_had_error : la tx précédente sur la même carte avait une erreur

    Tri indispensable par (card_id, _ts). Si card_id ou _ts manquants, on renvoie NaN/0.
    """
    if 'card_id' not in df.columns or df['_ts'].isna().all():
        df['time_since_last_trans_min']   = np.nan
        df['amount_last_24h']             = 0.0
        df['n_tx_last_24h']               = 0
        df['previous_transaction_had_error'] = 0
        return df

    # IMPORTANT : on conserve l'index d'origine pour pouvoir remettre les features
    # dans l'ordre original du DataFrame en fin de fonction.
    original_index = df.index.copy()
    df = df.sort_values(['card_id', '_ts'], kind='mergesort')

    # --- time_since_last_trans_min ---
    df['time_since_last_trans_min'] = (
        df.groupby('card_id', sort=False)['_ts'].diff().dt.total_seconds() / 60.0
    ).astype(np.float32)
    df['time_since_last_trans_min'] = df['time_since_last_trans_min'].replace(
        [np.inf, -np.inf], np.nan
    )

    # --- amount_last_24h / n_tx_last_24h (rolling time-based, closed='left' pour exclure la tx courante) ---
    roll = (
        df.set_index('_ts')
          .groupby('card_id', sort=False)['amount_abs']
          .rolling('24h', closed='left')
    )
    rolling_sum   = roll.sum().reset_index(level=0, drop=True)
    rolling_count = roll.count().reset_index(level=0, drop=True)

    # L'index de rolling_* est _ts (potentiels doublons) ; on se réaligne par position
    # car df est déjà trié exactement dans le même ordre (card_id, _ts).
    df['amount_last_24h'] = rolling_sum.values.astype(np.float32)
    df['n_tx_last_24h']   = rolling_count.values
    df['amount_last_24h'] = df['amount_last_24h'].fillna(0.0).astype(np.float32)
    df['n_tx_last_24h']   = df['n_tx_last_24h'].fillna(0).astype(np.int32)

    # --- previous_transaction_had_error : shift(1) du flag has_error, par card ---
    has_err = df.get('has_error', pd.Series(0, index=df.index))
    df['previous_transaction_had_error'] = (
        df.assign(_he=has_err)
          .groupby('card_id', sort=False)['_he']
          .shift(1)
          .fillna(0)
          .astype(np.int8)
    )

    # Remise dans l'ordre d'origine pour ne pas perturber un appel ultérieur
    df = df.reindex(original_index)
    return df

In [ ]:
def _financial_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Étape 5 — Ratios normalisés par le profil client (généralisables aux clients inconnus)."""
    if {'amount_abs', 'credit_limit'}.issubset(df.columns):
        df['ratio_amount_credit_limit'] = safe_divide(df['amount_abs'], df['credit_limit'])
    else:
        df['ratio_amount_credit_limit'] = np.nan

    if {'amount_abs', 'yearly_income'}.issubset(df.columns):
        monthly = df['yearly_income'] / 12.0
        df['ratio_amount_monthly_income'] = safe_divide(df['amount_abs'], monthly)
    else:
        df['ratio_amount_monthly_income'] = np.nan

    # Bonus utile : debt_to_income (profil financier du client)
    if {'total_debt', 'yearly_income'}.issubset(df.columns):
        df['debt_to_income'] = safe_divide(df['total_debt'], df['yearly_income'])
    else:
        df['debt_to_income'] = np.nan

    return df

In [ ]:
def _golden_features(df: pd.DataFrame) -> pd.DataFrame:
    """Étape 6 — Golden features (signaux empiriques très discriminants).

      - is_micro_transaction : montant absolu < 2$ (tests de carte typiques)
      - is_round_amount      : montant sans centimes (pattern fréquent de fraude automatisée)
      - amount_log1p         : log-transform pour les modèles linéaires / TabNet
    """
    if 'amount_abs' in df.columns:
        df['is_micro_transaction'] = (df['amount_abs'] < 2.0).astype(np.int8)
        # Arrondi au cent le plus proche pour éviter les imprécisions float
        cents = (df['amount_abs'] * 100).round().astype('Int64')
        df['is_round_amount'] = (cents % 100 == 0).astype(np.int8)
        df['amount_log1p'] = np.log1p(df['amount_abs']).astype(np.float32)
    else:
        df['is_micro_transaction'] = 0
        df['is_round_amount']      = 0
        df['amount_log1p']         = np.nan

    return df

In [ ]:
def _finalize_types_and_drop(df: pd.DataFrame) -> pd.DataFrame:
    """Étape 7/8 — Typage final + suppression des colonnes interdites/inutiles."""

    # 1) Drop des identifiants (anti-leakage)
    df = df.drop(columns=[c for c in ID_COLS if c in df.columns], errors='ignore')

    # 2) Drop des colonnes brutes
    df = df.drop(columns=[c for c in RAW_COLS_TO_DROP if c in df.columns], errors='ignore')
    df = df.drop(columns=['_ts'], errors='ignore')

    # 3) Typage numérique : tout ce qui est numérique -> float32 (sauf booléens int8 et cible int)
    keep_int_cols = {'is_fraud', 'is_weekend', 'is_night', 'is_online_tx',
                     'is_out_of_state', 'has_chip', 'card_on_dark_web', 'has_error',
                     'is_refund', 'is_micro_transaction', 'is_round_amount',
                     'previous_transaction_had_error', 'n_tx_last_24h',
                     'hour', 'dayofweek', 'month',
                     'num_credit_cards', 'num_cards_issued', 'credit_score',
                     'current_age', 'retirement_age', 'year_pin_last_changed'}

    num_cols = df.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        if c in keep_int_cols:
            # Garder un type entier compact
            try:
                df[c] = df[c].astype('Int32') if df[c].isna().any() else df[c].astype(np.int32)
            except Exception:
                df[c] = df[c].astype(np.float32)
        else:
            df[c] = df[c].astype(np.float32)

    # 4) Typage catégoriel (indispensable pour CatBoost / LightGBM auto-detection)
    for c in CATEGORICAL_COLS:
        if c in df.columns:
            df[c] = df[c].astype(str).fillna('missing').astype('category')

    # 5) is_fraud en tête de liste si présent (lisibilité)
    if 'is_fraud' in df.columns:
        cols = ['is_fraud'] + [c for c in df.columns if c != 'is_fraud']
        df = df[cols]

    return df

---
## 4. Fonction de pipeline globale

In [ ]:
def preprocess_dataset(df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    """Applique tout le pipeline de préparation à un DataFrame et renvoie la version prête à l'entraînement.

    Les étapes sont :
      1. Nettoyage de base ($ -> float, Yes/No -> 0/1, date -> datetime, errors -> lower)
      2. Features temporelles
      3. Features géographiques (home_state, is_out_of_state)
      4. Velocity features (sur card_id)
      5. Ratios financiers
      6. Golden features
      7. Typage final + drop des identifiants

    La fonction est IDEMPOTENTE (applicable plusieurs fois sans casser le résultat).
    """
    df = df.copy()
    steps = [
        ('01_basic_cleaning',    _basic_cleaning),
        ('02_time_features',     _time_features),
        ('03_geo_features',      _geo_features),
        ('04_velocity_features', _velocity_features),
        ('05_financial_ratios',  _financial_ratios),
        ('06_golden_features',   _golden_features),
        ('07_finalize',          _finalize_types_and_drop),
    ]
    for name, fn in steps:
        t0 = time.perf_counter()
        df = fn(df)
        if verbose:
            print(f'  [{name}] done in {time.perf_counter()-t0:.2f}s '
                  f'| shape={df.shape} | mem={memory_mb(df):.2f} MB')
    return df

---
## 5. Test du pipeline sur un seul fichier (sanity check)

In [ ]:
sample_file = 'train_010.0_pct.parquet'
df_raw = pd.read_parquet(sample_file)
print(f'RAW  : {df_raw.shape}  |  mem = {memory_mb(df_raw):.2f} MB')

df_prep = preprocess_dataset(df_raw, verbose=True)
print(f'\nPREP : {df_prep.shape}  |  mem = {memory_mb(df_prep):.2f} MB')
print('\nColonnes finales (dtypes) :')
print(df_prep.dtypes.astype(str).sort_values().to_string())

print('\nAperçu :')
df_prep.head(3)

In [ ]:
# Audit rapide : valeurs manquantes et distribution de la cible
print('Taux de fraude (ce fichier) :', f"{df_prep['is_fraud'].mean()*100:.3f} %")

na_report = df_prep.isna().mean().sort_values(ascending=False)
na_report = na_report[na_report > 0]
print('\nColonnes avec des NaN :')
print(na_report.to_string() if len(na_report) else '  (aucune)')

print('\nColonnes catégorielles (seront auto-détectées par CatBoost/LightGBM) :')
print([c for c, dt in df_prep.dtypes.items() if str(dt) == 'category'])

---
## 6. Batch processing — pipeline sur tous les fichiers

In [ ]:
def list_parquet_files(input_dir: Path, prefix_to_skip: str = OUTPUT_PREFIX) -> list:
    """Liste tous les .parquet d'entrée (train_*.parquet, test_*.parquet).
    Ignore les fichiers déjà préparés (prefixe `prepared_`)."""
    files = sorted([Path(f) for f in glob.glob(str(input_dir / '*.parquet'))])
    files = [f for f in files if not f.name.startswith(prefix_to_skip)]
    return files


files_to_process = list_parquet_files(INPUT_DIR)
print(f'{len(files_to_process)} fichiers à traiter :')
for f in files_to_process:
    size_mb = f.stat().st_size / 1024**2
    print(f'  - {f.name:<35}  ({size_mb:6.2f} MB sur disque)')

In [ ]:
def process_and_save(input_path: Path, output_dir: Path,
                     output_prefix: str = OUTPUT_PREFIX) -> dict:
    """Charge, transforme et sauvegarde un fichier. Retourne un dict de stats."""
    t0 = time.perf_counter()
    df_raw = pd.read_parquet(input_path)
    raw_shape = df_raw.shape
    raw_mem   = memory_mb(df_raw)

    df_prep = preprocess_dataset(df_raw, verbose=False)

    out_path = output_dir / f'{output_prefix}{input_path.name}'
    df_prep.to_parquet(out_path, index=False, compression='snappy')

    fraud_rate = float(df_prep['is_fraud'].mean()) if 'is_fraud' in df_prep.columns else np.nan

    return {
        'input_file':    input_path.name,
        'output_file':   out_path.name,
        'n_rows':        df_prep.shape[0],
        'n_cols_raw':    raw_shape[1],
        'n_cols_prep':   df_prep.shape[1],
        'mem_raw_mb':    round(raw_mem, 2),
        'mem_prep_mb':   round(memory_mb(df_prep), 2),
        'fraud_rate_%':  round(fraud_rate * 100, 3),
        'elapsed_s':     round(time.perf_counter() - t0, 2),
    }

In [ ]:
stats = []
iterator = tqdm(files_to_process, desc='Preparing datasets') if HAS_TQDM else files_to_process

for f in iterator:
    try:
        result = process_and_save(f, OUTPUT_DIR, OUTPUT_PREFIX)
        stats.append(result)
        if not HAS_TQDM:
            print(f"[OK] {result['input_file']:<32} -> {result['output_file']:<42} "
                  f"| {result['n_rows']:>7,} rows | {result['n_cols_prep']:>3} cols "
                  f"| {result['mem_prep_mb']:>6.2f} MB | fraud={result['fraud_rate_%']:>6.3f}% "
                  f"| {result['elapsed_s']:>5.2f}s")
    except Exception as e:
        print(f'[ERREUR] {f.name} : {e}')
        raise

stats_df = pd.DataFrame(stats)
print('\n=== Résumé du batch processing ===')
print(stats_df.to_string(index=False))

---
## 7. Vérification finale — relecture d'un fichier préparé

On relit le fichier `prepared_train_010.0_pct.parquet` et on vérifie que :
- Aucune colonne identifiant n'a survécu (anti-leakage).
- Les catégorielles sont bien typées `category`.
- Les numériques sont en `float32` / `Int32`.
- La cible `is_fraud` est présente.

In [ ]:
check_path = OUTPUT_DIR / f'{OUTPUT_PREFIX}train_010.0_pct.parquet'
df_check = pd.read_parquet(check_path)
print(f'Fichier relu : {check_path.name}  |  shape={df_check.shape}  |  mem={memory_mb(df_check):.2f} MB')

forbidden = [c for c in ID_COLS if c in df_check.columns]
assert len(forbidden) == 0, f'⚠ identifiants non supprimés : {forbidden}'
assert 'is_fraud' in df_check.columns, '⚠ colonne cible is_fraud absente'

print('\nIdentifiants supprimés       : OK')
print('Cible is_fraud présente      : OK')
print(f'Taux de fraude               : {df_check["is_fraud"].mean()*100:.3f} %')

cat_cols = [c for c, dt in df_check.dtypes.items() if str(dt) == 'category']
num_cols = df_check.select_dtypes(include=[np.number]).columns.tolist()
print(f'\nCatégorielles ({len(cat_cols)}) : {cat_cols}')
print(f'Numériques    ({len(num_cols)}) : {num_cols}')

print('\nDtypes :')
print(df_check.dtypes.astype(str).sort_values().to_string())

print('\nAperçu :')
df_check.head(5)

---
## 8. Bilan & prochaine étape

À la sortie de ce notebook, on dispose de :
- `prepared_train_XXX.X_pct.parquet` (10 fichiers) : datasets d'entraînement avec différents ratios d'undersampling.
- `prepared_test_050.0_pct.parquet` : dataset de test prêt à l'évaluation.

**Caractéristiques garanties** :
- ✅ Aucun identifiant client/card/merchant → le modèle est **forcé de généraliser**.
- ✅ Features **velocity** (`time_since_last_trans_min`, `amount_last_24h`, …) calculables sur un nouveau client dès sa 2ᵉ transaction.
- ✅ Ratios **normalisés par le profil client** → robustes aux nouveaux profils.
- ✅ Home-state dérivé des coordonnées GPS (pas d'historique) → généralisable.
- ✅ Types optimisés : numériques `float32`, catégorielles `category` (CatBoost/LightGBM les détectent automatiquement).

**Prochaine étape — notebook `03_Model_Training.ipynb`** :
1. Charger `prepared_train_XXX_pct.parquet` et `prepared_test_050.0_pct.parquet`.
2. Définir `cat_features = df.select_dtypes('category').columns.tolist()`.
3. Entraîner **CatBoost** (natif sur les catégorielles), **XGBoost** (avec `enable_categorical=True`) et **TabNet** (avec LabelEncoder + StandardScaler).
4. Évaluer avec **PR-AUC**, **ROC-AUC**, **F1**, **Recall@précision fixée** — métriques adaptées au fort déséquilibre.
5. Choisir le ratio d'undersampling qui maximise la métrique cible sur le test.